# Automatyzacja przetwarzania danych z użyciem Pipeline na zbiorze danych Titanic

**Mateusz Michałek 426994**

## 1. Wczytanie danych i podział ze stratyfikacją

In [17]:

# import niezbędnych pakietów
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv("titanic.csv")

cat_features = ["cabin", "CabinReduced", "sex"]
X = df[cat_features].copy()
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Wielkość zbioru treningowego: {X_train.shape}, wielkość zbioru testowego: {X_test.shape}")

Wielkość zbioru treningowego: (1047, 3), wielkość zbioru testowego: (262, 3)


## 2. Budowa Pipeline dla zmiennych kategorycznych

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_features)
    ]
)

* **Czy wystarczy nam tutaj użycie OneHotEncoder?**
   OneHotEncoder dobrze sprawdza się dla zmiennych o niskiej kardynalności (`sex`, `CabinReduced`), ale dla cechy `cabin` o wysokiej kardynalności tworzy kilkaset rzadkich kolumn binarnych, więc lepiej zastosować redukcję cechy (np. `CabinReduced`) lub inne kodowanie.
* **Co robi handle_unknown="ignore" w OneHotEncoder?**
  Zapobiega zgłoszeniu błędu `ValueError`, gdy w `X_test` pojawi się kategoria niewystępująca w `X_train` – taka nieznana etykieta kodowana jest jako wektor samych zer `[0, 0, ..., 0]`.
* **Dlaczego warto użyć ColumnTransformer zamiast osobnego Pipeline dla każdej kolumny?**
  `ColumnTransformer` łączy transformacje różnych podzbiorów kolumn (np. kategorycznych i numerycznych) w jeden spójny obiekt, wykonuje je równolegle bez ręcznego rozbijania i sklejania ramki `DataFrame` oraz eliminuje ryzyko wycieku danych (*data leakage*).

# 3. Budowa modelu predykcji - trening i ewaluacja

In [14]:
model = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(random_state=42))
])

model.fit(X_train, y_train)
acc = model.score(X_test, y_test)
print(f'Dokładność: {acc:.3f}')

Dokładność: 0.794


## 4. Schemat Pipeline oraz weryfikacja przetworzonych danych

In [15]:
set_config(display='diagram')  # pokazuje schemat pipeline
model

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['cabin', 'CabinReduced',
                                                   'sex'])])),
                ('clf', RandomForestClassifier(random_state=42))])

In [16]:
# 1. Sprawdzenie braków przed i po kroku SimpleImputer
imputer = model.named_steps["prep"].named_transformers_["cat"].named_steps["imputer"]
X_train_imp = pd.DataFrame(
    imputer.transform(X_train), columns=cat_features, index=X_train.index
)
X_test_imp = pd.DataFrame(
    imputer.transform(X_test), columns=cat_features, index=X_test.index
)

print("Braki w X_train przed imputacją:\n", X_train.isnull().sum())
print("Braki w X_test przed imputacją:\n", X_test.isnull().sum())
print("\nBraki w X_train po imputacji:\n", X_train_imp.isnull().sum())
print("Braki w X_test po imputacji:", X_test_imp.isnull().sum().sum())

# 2. Liczba nowych kolumn po OneHotEncoder
X_train_encoded = model.named_steps["prep"].transform(X_train)
ohe = model.named_steps["prep"].named_transformers_["cat"].named_steps[
    "encoder"
]
feature_names = ohe.get_feature_names_out(cat_features)

print("\nKształt macierzy po OneHotEncoder (X_train):", X_train_encoded.shape)
for col, cats in zip(cat_features, ohe.categories_):
    print(f"Liczba kolumn binarnych dla '{col}': {len(cats)}")

# 3. Sprawdzenie nowych kategorii w zbiorze testowym
print("\nNowe (niewidziane w X_train) kategorie w X_test:")
for col in cat_features:
    train_cats = set(X_train[col].dropna().unique())
    test_cats = set(X_test[col].dropna().unique())
    unseen = test_cats - train_cats
    print(f" - {col}: {len(unseen)} nowych kategorii")

Braki w X_train przed imputacją:
 cabin           822
CabinReduced    822
sex               0
dtype: int64
Braki w X_test przed imputacją:
 cabin           192
CabinReduced    192
sex               0
dtype: int64

Braki w X_train po imputacji:
 cabin           0
CabinReduced    0
sex             0
dtype: int64
Braki w X_test po imputacji: 0

Kształt macierzy po OneHotEncoder (X_train): (1047, 165)
Liczba kolumn binarnych dla 'cabin': 155
Liczba kolumn binarnych dla 'CabinReduced': 8
Liczba kolumn binarnych dla 'sex': 2

Nowe (niewidziane w X_train) kategorie w X_test:
 - cabin: 31 nowych kategorii
 - CabinReduced: 0 nowych kategorii
 - sex: 0 nowych kategorii


1. **Sprawdź, czy liczba braków została poprawnie uzupełniona.**
Wszystkie braki w kolumnie `cabin` (822 w `X_train` oraz 192 w `X_test`) zostały poprawnie uzupełnione przez `SimpleImputer` (liczba `NaN` po transformacji wynosi `0` w obu zbiorach). Warto jednak pamiętać, że strategia `most_frequent` przy ~77% braków przypisuje większości pasażerów tę samą najczęstszą kabinę.
2. **Czy liczba nowych kolumn po OneHotEncoder jest zgodna z oczekiwaniami? Czy OneHotEncoder był wystarczający?**
Liczba wygenerowanych kolumn binarnych dokładnie odpowiada sumie unikalnych kategorii występujących w `X_train` po imputacji (2 kolumny dla `sex`, 8 dla `CabinReduced` oraz 155 dla `cabin`). Potwierdza to, że `OneHotEncoder` jest w pełni wystarczający dla `sex` i `CabinReduced`, lecz dla nieprzetworzonej cechy `cabin` prowadzi do nadmiernej liczby rzadkich kolumn.
3. **Co by się stało, gdyby w zbiorze testowym pojawiła się nowa kategoria? Pojawiła się w tym przypadku?**
Dla `sex` oraz `CabinReduced` w `X_test` nie pojawiły się żadne nowe kategorie (`0`). Natomiast dla zmiennej `cabin` w `X_test` wystąpiło kilkadziesiąt nowych numerów kabin niewystępujących w `X_train`. Bez `handle_unknown="ignore"` kodowanie zakończyłoby się błędem `ValueError`, dzięki temu parametrowi nieznane kabiny zostały bezpiecznie zakodowane jako wektory zer.

In [ ]:
df = 